# API Rundown

Basically, we have 
```python
torch.utils.data.DataLoader
torch.utils.data.Dataset
```

`torch.utils.data.Dataset` saves our samples of the dataset and their corresponding labels. `torch.utils.data.DataLoader` is just a iterable wrapper around the `Dataset`

In [38]:
import torch
from torch import nn
# from torch.utils.data import Dataset
from torch.utils.data import DataLoader
from torchvision.transforms import ToTensor
from torchvision import datasets

For this API rundown we will use the FashionMNIST dataset

In [39]:
trainingData = datasets.FashionMNIST(
    root= "data",
    train = True,
    download = True,
    transform = ToTensor())

In [40]:
testData = datasets.FashionMNIST(root = "data",
                                 train = False,
                                 download = True,
                                 transform = ToTensor())

In [41]:
batchSize = 64

trainDataLoader = DataLoader(trainingData, batch_size=batchSize)
testDataLoader = DataLoader(testData, batch_size=batchSize)

In [42]:
for X, y in testDataLoader:
    print(f"Shape of X[N, C, H, W]: {X.shape}")
    print(f"Shape of y: {y.shape}")
    break
# Printing for one batch, break because every batch has the same shape for features and labels

Shape of X[N, C, H, W]: torch.Size([64, 1, 28, 28])
Shape of y: torch.Size([64])


In [43]:
device = torch.accelerator.current_accelerator().type if torch.accelerator.is_available() else "cpu"

print(f"Using device: {device}")

Using device: cpu


---

In [44]:
class NeuralNetwork(nn.Module):
    def __init__(self):
        super().__init__()
        self.flatten = nn.Flatten()
        self.linearReLUStack = nn.Sequential(
            nn.Linear(28 * 28, 512),
            nn.ReLU(),
            nn.Linear(512, 512),
            nn.ReLU(),
            nn.Linear(512, 10),
        )
    def forward(self, x):  # Ouput of NN before applyting activation function like softmax
        x = self.flatten(x)
        logits = self.linearReLUStack(x)
        return logits

model = NeuralNetwork().to(device)
print(model)

NeuralNetwork(
  (flatten): Flatten(start_dim=1, end_dim=-1)
  (linearReLUStack): Sequential(
    (0): Linear(in_features=784, out_features=512, bias=True)
    (1): ReLU()
    (2): Linear(in_features=512, out_features=512, bias=True)
    (3): ReLU()
    (4): Linear(in_features=512, out_features=10, bias=True)
  )
)


In [45]:
lossFunction = nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(model.parameters(), lr = 0.0001)

In [46]:
def train(dataloader, model, lossFunction, optimizer):
    size = len(dataloader.dataset)
    model.train()

    for batch, (X,y) in enumerate(dataloader):
        X,y = X.to(device), y.to(device)

        #Prediction error
        pred = model(X)
        loss = lossFunction(pred, y)

        #Backprop
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()

        if batch % 100 == 0:
            loss, current = loss.item(), (batch + 1) * len(X)
            print(f"Loss: {loss} [{current}/{size}]")
        

In [47]:
def test(dataloader, model, loss_fn):
    size = len(dataloader.dataset)
    num_batches = len(dataloader)
    model.eval()
    test_loss, correct = 0, 0
    with torch.no_grad():
        for X, y in dataloader:
            X, y = X.to(device), y.to(device)
            pred = model(X)
            test_loss += loss_fn(pred, y).item()
            correct += (pred.argmax(1) == y).type(torch.float).sum().item()
    test_loss /= num_batches
    correct /= size
    print(f"Test Error: \n Accuracy: {(100*correct):>0.1f}%, Avg loss: {test_loss:>8f} \n")

In [48]:
epochs = 5
for t in range(epochs):
    print(f"Epoch {t+1}\n-------------------------------")
    train(trainDataLoader, model, lossFunction, optimizer)
    test(testDataLoader, model, lossFunction)
print("Done!")

Epoch 1
-------------------------------
Loss: 2.3058221340179443 [64/60000]
Loss: 2.295567512512207 [6464/60000]
Loss: 2.2943320274353027 [12864/60000]
Loss: 2.2862937450408936 [19264/60000]
Loss: 2.3033385276794434 [25664/60000]
Loss: 2.29276442527771 [32064/60000]
Loss: 2.2929320335388184 [38464/60000]
Loss: 2.295351028442383 [44864/60000]
Loss: 2.2916581630706787 [51264/60000]
Loss: 2.2853639125823975 [57664/60000]
Test Error: 
 Accuracy: 24.8%, Avg loss: 2.284886 

Epoch 2
-------------------------------
Loss: 2.2933907508850098 [64/60000]
Loss: 2.2844605445861816 [6464/60000]
Loss: 2.280787944793701 [12864/60000]
Loss: 2.2756171226501465 [19264/60000]
Loss: 2.290227174758911 [25664/60000]
Loss: 2.2775650024414062 [32064/60000]
Loss: 2.2816002368927 [38464/60000]
Loss: 2.2813801765441895 [44864/60000]
Loss: 2.279714345932007 [51264/60000]
Loss: 2.272102117538452 [57664/60000]
Test Error: 
 Accuracy: 29.0%, Avg loss: 2.271827 

Epoch 3
-------------------------------
Loss: 2.2812557